# Part 1: Data Audit, EDA & Business Understanding

**Objective:** Before building any model or retention strategy, we need to understand
our data inside out — what we have, what's broken, what patterns exist, and what
hypotheses we can form about churn.

**Dataset:** D2C personal-care brand with 2,400 customers. Snapshot date: 2025-09-30.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
customers = pd.read_csv('data/customers.csv')
orders = pd.read_csv('data/orders.csv')
support_tickets = pd.read_csv('data/support_tickets.csv')
web_events = pd.read_csv('data/web_events_snapshot.csv')
churn_labels = pd.read_csv('data/churn_labels.csv')
rfm_snapshot = pd.read_csv('data/rfm_modeling_snapshot.csv')
interventions = pd.read_csv('data/intervention_history.csv')

In [3]:
customers.info()
print("📊 All datasets loaded! Here are their sizes:\n")
datasets = {
    'customers': customers,
    'orders': orders,
    'support_tickets': support_tickets,
    'web_events': web_events,
    'churn_labels': churn_labels,
    'rfm_snapshot': rfm_snapshot,
    'interventions': interventions
}
for name, df in datasets.items():
    print(f"  {name:20s} → {df.shape[0]:>5} rows × {df.shape[1]:>2} columns")

print("📋 Columns in each dataset:\n")
for name, df in datasets.items():
    print(f"  {name}:")
    print(f"    → {list(df.columns)}\n")
print("First 5 orders:")
print(orders.head())
print(f"\nColumn types in orders:")
print(orders.dtypes)

<class 'pandas.DataFrame'>
RangeIndex: 2400 entries, 0 to 2399
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   customer_id          2400 non-null   str  
 1   signup_date          2400 non-null   str  
 2   city_tier            2400 non-null   str  
 3   age_group            2400 non-null   str  
 4   acquisition_channel  2400 non-null   str  
 5   loyalty_tier         1014 non-null   str  
 6   preferred_category   2400 non-null   str  
 7   skin_type            1999 non-null   str  
 8   marketing_consent    2400 non-null   str  
dtypes: str(9)
memory usage: 168.9 KB
📊 All datasets loaded! Here are their sizes:

  customers            →  2400 rows ×  9 columns
  orders               → 10009 rows × 10 columns
  support_tickets      →  1921 rows ×  8 columns
  web_events           →  2400 rows × 10 columns
  churn_labels         →  2400 rows ×  4 columns
  rfm_snapshot         →  2400 rows × 29 columns

In [4]:
print("🔗 Unique customer_id counts:\n")
for name, df in datasets.items():
    unique_count = df['customer_id'].nunique()
    total_rows = len(df)
    if unique_count == total_rows:
        relationship = "one-to-one (1 row per customer)"
    elif unique_count < total_rows:
        avg_per_customer = total_rows / unique_count
        relationship = f"one-to-many (~{avg_per_customer:.1f} rows per customer)"
    print(f"  {name:20s} → {unique_count:>5} unique customers out of {total_rows:>5} rows  [{relationship}]")

🔗 Unique customer_id counts:

  customers            →  2400 unique customers out of  2400 rows  [one-to-one (1 row per customer)]
  orders               →  2400 unique customers out of 10009 rows  [one-to-many (~4.2 rows per customer)]
  support_tickets      →  1247 unique customers out of  1921 rows  [one-to-many (~1.5 rows per customer)]
  web_events           →  2400 unique customers out of  2400 rows  [one-to-one (1 row per customer)]
  churn_labels         →  2400 unique customers out of  2400 rows  [one-to-one (1 row per customer)]
  rfm_snapshot         →  2400 unique customers out of  2400 rows  [one-to-one (1 row per customer)]
  interventions        →  2400 unique customers out of  2400 rows  [one-to-one (1 row per customer)]


In [5]:
# Checking missing values in EVERY dataset
print("🔍 Missing values across all datasets:\n")
for name, df in datasets.items():
    missing = df.isnull().sum()
    missing_only = missing[missing > 0]
    if len(missing_only) == 0:
        print(f"  {name}: ✅ No missing values!")
    else:
        print(f"  {name}:")
        for col, count in missing_only.items():
            pct = count / len(df) * 100
            print(f"    → {col}: {count} missing ({pct:.1f}%)")
    print()

🔍 Missing values across all datasets:

  customers:
    → loyalty_tier: 1386 missing (57.8%)
    → skin_type: 401 missing (16.7%)

  orders:
    → rating: 80 missing (0.8%)

  support_tickets: ✅ No missing values!

  web_events: ✅ No missing values!

  churn_labels: ✅ No missing values!

  rfm_snapshot:
    → loyalty_tier: 1386 missing (57.8%)

  interventions: ✅ No missing values!



In [6]:
# Find orders where order_id ends with '_DUP'
dup_mask = orders['order_id'].str.endswith('_DUP')
dup_orders = orders[dup_mask]

print(f" Duplicate orders found: {len(dup_orders)}")
print(f"   Out of {len(orders)} total orders ({len(dup_orders)/len(orders)*100:.1f}%)\n")
print("First 5 duplicate orders:")
print(dup_orders.head())

 Duplicate orders found: 12
   Out of 10009 total orders (0.1%)

First 5 duplicate orders:
           order_id customer_id  order_date   category  quantity  \
601   ORD008249_DUP   CUST00153  2025-11-04  Hair Care         1   
2621  ORD002124_DUP   CUST00628  2025-03-18  Skin Care         1   
3534  ORD002862_DUP   CUST00837  2025-07-12  Hair Care         3   
3602  ORD002916_DUP   CUST00848  2025-09-26  Skin Care         1   
3675  ORD002970_DUP   CUST00869  2024-12-22  Fragrance         1   

      gross_amount  discount_pct  delivery_days  returned  rating  
601         321.31          0.36              8         0     3.0  
2621        410.04          0.47              3         0     5.0  
3534        952.02          0.47              4         0     4.0  
3602        547.18          0.28              2         0     5.0  
3675        818.64          0.18              2         0     4.0  


In [7]:
# Get the original order IDs by removing '_DUP' from the duplicate IDs
dup_original_ids = dup_orders['order_id'].str.replace('_DUP', '', regex=False)

# Find those originals in the orders table
originals = orders[orders['order_id'].isin(dup_original_ids)]

print(f"Found {len(originals)} matching original orders for {len(dup_orders)} duplicates\n")

# Let's compare the first duplicate with its original
print("Example comparison — first duplicate vs its original:\n")
first_dup = dup_orders.iloc[0]  # First duplicate row
first_dup_id = first_dup['order_id']
original_id = first_dup_id.replace('_DUP', '')
first_original = orders[orders['order_id'] == original_id].iloc[0]

# Show them side by side
print(f"  {'Column':<20} {'Original':>15} {'Duplicate':>15} {'Match?':>8}")
print(f"  {'─'*60}")
for col in orders.columns:
    orig_val = first_original[col]
    dup_val = first_dup[col]
    match = "✅" if str(orig_val) == str(dup_val) or col == 'order_id' else "❌"
    print(f"  {col:<20} {str(orig_val):>15} {str(dup_val):>15} {match:>8}")

Found 12 matching original orders for 12 duplicates

Example comparison — first duplicate vs its original:

  Column                      Original       Duplicate   Match?
  ────────────────────────────────────────────────────────────
  order_id                   ORD008249   ORD008249_DUP        ✅
  customer_id                CUST00153       CUST00153        ✅
  order_date                2025-11-04      2025-11-04        ✅
  category                   Hair Care       Hair Care        ✅
  quantity                           1               1        ✅
  gross_amount                  321.31          321.31        ✅
  discount_pct                    0.36            0.36        ✅
  delivery_days                      8               8        ✅
  returned                           0               0        ✅
  rating                           3.0             3.0        ✅


In [8]:
# Remove duplicate orders
orders_clean = orders[~dup_mask].copy()

print(f"Before: {len(orders)} orders")
print(f"Removed: {len(dup_orders)} duplicates")
print(f"After:  {len(orders_clean)} orders ✅")

Before: 10009 orders
Removed: 12 duplicates
After:  9997 orders ✅


In [9]:
# Converting order_date from text to datetime
orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'])

# Defining provided snapshot date
snapshot_date = pd.to_datetime('2025-09-30')

# How many orders are BEFORE vs AFTER the snapshot?
pre_snapshot = orders_clean[orders_clean['order_date'] <= snapshot_date]
post_snapshot = orders_clean[orders_clean['order_date'] > snapshot_date]

print(f"   Snapshot date: {snapshot_date.date()}")
print(f"   Orders ON or BEFORE snapshot: {len(pre_snapshot)} (safe to use as features)")
print(f"   Orders AFTER snapshot:        {len(post_snapshot)} ( FUTURE DATA — cannot use as features)")
print(f"   Date range of post-snapshot:  {post_snapshot['order_date'].min().date()} to {post_snapshot['order_date'].max().date()}")


   Snapshot date: 2025-09-30
   Orders ON or BEFORE snapshot: 8128 (safe to use as features)
   Orders AFTER snapshot:        1869 ( FUTURE DATA — cannot use as features)
   Date range of post-snapshot:  2025-10-01 to 2025-11-29


In [10]:
# Looking at the statistics of gross_amount (using pre-snapshot orders only)
print(" gross_amount statistics (pre-snapshot orders):\n")
print(pre_snapshot['gross_amount'].describe())

# Also checking the top 10 highest orders
print(f"\n\n🔝 Top 10 highest gross_amount orders:")
top_orders = pre_snapshot.nlargest(10, 'gross_amount')[['order_id', 'customer_id', 'gross_amount', 'category', 'quantity']]
print(top_orders.to_string(index=False))

 gross_amount statistics (pre-snapshot orders):

count     8128.000000
mean       752.101426
std        626.461692
min        149.000000
25%        431.772500
50%        598.130000
75%        922.922500
max      24789.380000
Name: gross_amount, dtype: float64


🔝 Top 10 highest gross_amount orders:
 order_id customer_id  gross_amount  category  quantity
ORD006374   CUST01868      24789.38 Skin Care         3
ORD000701   CUST00211      22719.45 Fragrance         2
ORD007206   CUST02106      15957.48 Fragrance         2
ORD004428   CUST01295      10643.82 Baby Care         2
ORD004650   CUST01360       8777.20 Fragrance         2
ORD005399   CUST01584       8022.50 Fragrance         1
ORD007765   CUST02287       3746.76 Fragrance         4
ORD000500   CUST00159       3376.32 Fragrance         4
ORD001120   CUST00324       3341.27 Fragrance         4
ORD004146   CUST01214       3098.18 Fragrance         3


In [11]:
os.makedirs('charts', exist_ok=True)

# Let's verify it loaded correctly by printing the version
print(f"✅ pandas loaded successfully! Version: {pd.__version__}")
print(f"✅ matplotlib loaded! Version: {plt.matplotlib.__version__}")

✅ pandas loaded successfully! Version: 3.0.3
✅ matplotlib loaded! Version: 3.10.9


In [12]:
# Creating a box plot of gross_amount
fig, ax = plt.subplots(figsize=(10, 4))

ax.boxplot(pre_snapshot['gross_amount'], vert=False)
ax.set_xlabel('Gross Amount (₹)')
ax.set_title('Distribution of Order Amounts (Pre-Snapshot) — Outliers Visible')

# Saving it according to the guidelines in charts/ folder
plt.savefig('charts/gross_amount_outliers.png', dpi=150, bbox_inches='tight')
plt.close()

print("✅ Chart saved: charts/gross_amount_outliers.png")

✅ Chart saved: charts/gross_amount_outliers.png


In [13]:
# What is the overall churn rate from the data set?
print("Overall Churn Distribution:\n")
churn_counts = churn_labels['churn_next_60d'].value_counts()
print(churn_counts)

churn_rate = churn_labels['churn_next_60d'].mean() * 100
print(f"\n→ Churn rate: {churn_rate:.1f}%")
print(f"→ {churn_counts[1]} customers churned out of {len(churn_labels)}")
print(f"→ {churn_counts[0]} customers stayed")

# Also check the train/val/test split
print(f"\n Train/Val/Test split:")
print(churn_labels['split'].value_counts())

Overall Churn Distribution:

churn_next_60d
0    1273
1    1127
Name: count, dtype: int64

→ Churn rate: 47.0%
→ 1127 customers churned out of 2400
→ 1273 customers stayed

 Train/Val/Test split:
split
train         1728
validation     336
test           336
Name: count, dtype: int64


In [14]:
# Merge customers with churn labels
customer_churn = pd.merge(customers, churn_labels[['customer_id', 'churn_next_60d']], 
                           on='customer_id')

print(f"Merged table: {customer_churn.shape[0]} rows × {customer_churn.shape[1]} columns")
print(f"(Should be 2,400 rows — one per customer, now with churn label attached)\n")

# Churn rate by age group
print("Churn rate by AGE GROUP:")
churn_by_age = customer_churn.groupby('age_group')['churn_next_60d'].mean() * 100
print(churn_by_age.round(1).to_string())

# Churn rate by city tier
print("\n Churn rate by CITY TIER:")
churn_by_city = customer_churn.groupby('city_tier')['churn_next_60d'].mean() * 100
print(churn_by_city.round(1).to_string())

# Churn rate by acquisition channel
print("\n Churn rate by ACQUISITION CHANNEL:")
churn_by_channel = customer_churn.groupby('acquisition_channel')['churn_next_60d'].mean() * 100
print(churn_by_channel.round(1).to_string())

Merged table: 2400 rows × 10 columns
(Should be 2,400 rows — one per customer, now with churn label attached)

Churn rate by AGE GROUP:
age_group
18-24    45.5
25-34    47.2
35-44    48.3
45+      46.4

 Churn rate by CITY TIER:
city_tier
Tier 1    47.4
Tier 2    47.7
Tier 3    45.0

 Churn rate by ACQUISITION CHANNEL:
acquisition_channel
Google Search    50.4
Influencer       47.6
Instagram        49.9
Marketplace      49.1
Organic          39.8
Referral         42.2


In [15]:
# Creating 3 bar charts — one for each demographic dimension that we took in last step
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Chart 1: Churn by Age Group
churn_by_age.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].axhline(y=churn_rate, color='red', linestyle='--', label=f'Overall: {churn_rate:.1f}%')
axes[0].set_title('Churn Rate by Age Group')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_xlabel('')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=0)

# Chart 2: Churn by City Tier
churn_by_city.plot(kind='bar', ax=axes[1], color='darkorange')
axes[1].axhline(y=churn_rate, color='red', linestyle='--', label=f'Overall: {churn_rate:.1f}%')
axes[1].set_title('Churn Rate by City Tier')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_xlabel('')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=0)

# Chart 3: Churn by Acquisition Channel
churn_by_channel.sort_values().plot(kind='barh', ax=axes[2], color='seagreen')
axes[2].axvline(x=churn_rate, color='red', linestyle='--', label=f'Overall: {churn_rate:.1f}%')
axes[2].set_title('Churn Rate by Acquisition Channel')
axes[2].set_xlabel('Churn Rate (%)')
axes[2].set_ylabel('')
axes[2].legend()

plt.tight_layout()
plt.savefig('charts/demographics_vs_churn.png', dpi=150, bbox_inches='tight')
plt.close()

print("✅ Chart saved: charts/demographics_vs_churn.png")


✅ Chart saved: charts/demographics_vs_churn.png


## Hypothesis 1&2: Age and city tier are weak churn predictors
#### Customer demographics (age group, city tier) do NOT meaningfully predict churn.
#### **Evidence**: Age group churn ranges from 45.5% to 48.3% (only ~3% spread).
City tier ranges from 45.0% to 47.7% (only ~2.7% spread). Compare this to acquisition channel which has a 10.6% spread. This means targeting retention campaigns by age or city would be ineffective — behavioural signals are far more useful.

## Hypothesis 3: Acquisition channel affects retention
#### Organic and referral customers are more loyal than paid-channel customers.
#### **Evidence**: Organic churn = 39.8%, Referral = 42.2% vs Google Search = 50.4%
(see chart above). Organic/referral customers chose the brand themselves, while paid customers were "bought" and may not be committed.

In [16]:
# Count tickets per customer
tickets_per_customer = support_tickets.groupby('customer_id').size().reset_index(name='ticket_count')

# Merge with churn labels — LEFT join keeps ALL customers
customer_tickets_churn = pd.merge(
    churn_labels[['customer_id', 'churn_next_60d']], 
    tickets_per_customer, 
    on='customer_id', 
    how='left'  # Keep customers even if they have NO tickets
)

# Customers without tickets get NaN — fill with 0
customer_tickets_churn['ticket_count'] = customer_tickets_churn['ticket_count'].fillna(0).astype(int)

# Compare: customers WITH tickets vs WITHOUT
customer_tickets_churn['has_tickets'] = customer_tickets_churn['ticket_count'] > 0

print("Churn rate: Customers WITH tickets vs WITHOUT:\n")
churn_by_tickets = customer_tickets_churn.groupby('has_tickets')['churn_next_60d'].agg(['mean', 'count'])
churn_by_tickets.columns = ['churn_rate', 'customer_count']
churn_by_tickets['churn_rate'] = (churn_by_tickets['churn_rate'] * 100).round(1)
churn_by_tickets.index = ['No tickets', 'Has tickets']
print(churn_by_tickets)

Churn rate: Customers WITH tickets vs WITHOUT:

             churn_rate  customer_count
No tickets         47.2            1153
Has tickets        46.8            1247


## Hypothesis 4: Support tickets don't drive churn — but sentiment might
### Simply having support tickets doesn't predict churn (47.2% vs 46.8%).
#### **Evidence**: The table above shows almost no difference. ticket_count_90d has weak correlation (r = -0.153). However, ticket *sentiment* and resolution quality may matter — filing a complaint might actually show the customer still cares.

In [17]:
# Chart 3: Overall churn distribution — bar chart with counts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Churn count bar chart
churn_labels['churn_next_60d'].value_counts().plot(
    kind='bar', ax=axes[0], color=['seagreen', 'tomato'])
axes[0].set_title('Overall Churn Distribution')
axes[0].set_xlabel('Churn Status')
axes[0].set_ylabel('Number of Customers')
axes[0].set_xticklabels(['Stayed (0)', 'Churned (1)'], rotation=0)

# Annotate counts on bars
for i, v in enumerate(churn_labels['churn_next_60d'].value_counts()):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# Right: Churn rate by loyalty tier (including "Not Enrolled")
# Fill NaN loyalty_tier with "Not Enrolled" for this analysis
customer_churn_loyalty = customer_churn.copy()
customer_churn_loyalty['loyalty_tier'] = customer_churn_loyalty['loyalty_tier'].fillna('Not Enrolled')

churn_by_loyalty = customer_churn_loyalty.groupby('loyalty_tier')['churn_next_60d'].mean() * 100
# Order: Not Enrolled, Silver, Gold, Platinum
tier_order = ['Not Enrolled', 'Silver', 'Gold', 'Platinum']
churn_by_loyalty = churn_by_loyalty.reindex(tier_order)

churn_by_loyalty.plot(kind='bar', ax=axes[1], color=['gray', 'silver', 'gold', 'mediumpurple'])
axes[1].axhline(y=churn_rate, color='red', linestyle='--', label=f'Overall: {churn_rate:.1f}%')
axes[1].set_title('Churn Rate by Loyalty Tier')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_xlabel('')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('charts/churn_distribution_loyalty.png', dpi=150, bbox_inches='tight')
plt.close()

print(" Chart saved: charts/churn_distribution_loyalty.png")

# Print the loyalty tier numbers too
print("\n Churn rate by LOYALTY TIER:")
for tier in tier_order:
    rate = churn_by_loyalty[tier]
    count = len(customer_churn_loyalty[customer_churn_loyalty['loyalty_tier'] == tier])
    print(f"  {tier:15s} → {rate:.1f}% churn ({count} customers)")

 Chart saved: charts/churn_distribution_loyalty.png

 Churn rate by LOYALTY TIER:
  Not Enrolled    → 48.3% churn (1386 customers)
  Silver          → 48.8% churn (590 customers)
  Gold            → 40.8% churn (319 customers)
  Platinum        → 37.1% churn (105 customers)


## Hypothesis 5: Higher loyalty tiers reduce churn
### Platinum members churn less than Silver or non-enrolled customers.
#### **Evidence**: Platinum churn = 37.1%, Gold = 40.8%, Silver = 48.8%, Not Enrolled = 48.3% 
(see chart and table above). Clear downward trend with higher tiers — the loyalty program is working, but 58% aren't enrolled.

In [18]:
# Merge web events with churn labels
web_churn = pd.merge(web_events, churn_labels[['customer_id', 'churn_next_60d']], on='customer_id')

# Comparing the key web metrics between churners (1) and stayers (0)
web_cols = ['sessions_30d', 'product_views_30d', 'cart_adds_30d', 'last_visit_days_ago']

print("\n Web activity: Churners vs Stayers (averages):\n")
print(f"  {'Metric':<25} {'Stayed':>10} {'Churned':>10} {'Difference':>12}")
print(f"  {'─'*60}")
for col in web_cols:
    stayed_avg = web_churn[web_churn['churn_next_60d'] == 0][col].mean()
    churned_avg = web_churn[web_churn['churn_next_60d'] == 1][col].mean()
    diff = churned_avg - stayed_avg
    print(f"  {col:<25} {stayed_avg:>10.1f} {churned_avg:>10.1f} {diff:>+12.1f}")

# Chart 5: Web activity comparison — grouped bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Sessions and page views comparison
web_comparison = web_churn.groupby('churn_next_60d')[['sessions_30d', 'product_views_30d', 'cart_adds_30d']].mean()
web_comparison.index = ['Stayed', 'Churned']
web_comparison.plot(kind='bar', ax=axes[0], color=['steelblue', 'darkorange', 'seagreen'])
axes[0].set_title('Web Activity: Churners vs Stayers')
axes[0].set_ylabel('Average Count (30 days)')
axes[0].set_xlabel('')
axes[0].legend(title='Metric')
axes[0].tick_params(axis='x', rotation=0)

# Right: Last visit days ago (higher = less recent = more likely to churn)
web_churn['churn_label'] = web_churn['churn_next_60d'].map({0: 'Stayed', 1: 'Churned'})
web_churn.boxplot(column='last_visit_days_ago', by='churn_label', ax=axes[1])
axes[1].set_title('Days Since Last Visit: Churners vs Stayers')
axes[1].set_ylabel('Days Since Last Visit')
axes[1].set_xlabel('')
plt.suptitle('')

plt.tight_layout()
plt.savefig('charts/web_events_vs_churn.png', dpi=150, bbox_inches='tight')
plt.close()

print("\n✅ Chart saved: charts/web_events_vs_churn.png")


 Web activity: Churners vs Stayers (averages):

  Metric                        Stayed    Churned   Difference
  ────────────────────────────────────────────────────────────
  sessions_30d                     6.7        4.0         -2.7
  product_views_30d               28.4       17.0        -11.4
  cart_adds_30d                    1.9        1.1         -0.8
  last_visit_days_ago              9.8       26.6        +16.8

✅ Chart saved: charts/web_events_vs_churn.png


## Hypothesis 6: Web disengagement precedes churn
### Customers who stop visiting the website churn more.
#### **Evidence**: Churners average 26.6 days since last visit vs 9.8 for stayers
(see table and chart above). Churners also have 2.7 fewer sessions and 11.4 fewer product views. Web activity drops BEFORE they stop buying.

In [19]:
# Using rfm_snapshot since it has all features in one table && Selecting only numeric columns for correlation
numeric_cols = ['recency_days', 'frequency_180d', 'monetary_180d', 'return_rate_180d',
                'avg_discount_pct_180d', 'ticket_count_90d', 'sessions_30d', 
                'product_views_30d', 'last_visit_days_ago', 'churn_next_60d']

corr_matrix = rfm_snapshot[numeric_cols].corr()

# Chart 6: Correlation heatmap
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation Heatmap \u2014 Key Features vs Churn')

plt.tight_layout()
plt.savefig('charts/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()

# Print the most correlated features with churn
print("\n Correlation with churn_next_60d (sorted by strength):")
churn_corr = corr_matrix['churn_next_60d'].drop('churn_next_60d').sort_values(key=abs, ascending=False)
for feature, corr in churn_corr.items():
    direction = "↑ more churn" if corr > 0 else "↓ less churn"
    print(f"  {feature:<30} r = {corr:+.3f}  ({direction})")

print("\n✅ Chart saved: charts/correlation_heatmap.png")



 Correlation with churn_next_60d (sorted by strength):
  recency_days                   r = +0.561  (↑ more churn)
  last_visit_days_ago            r = +0.527  (↑ more churn)
  frequency_180d                 r = -0.394  (↓ less churn)
  monetary_180d                  r = -0.385  (↓ less churn)
  sessions_30d                   r = -0.307  (↓ less churn)
  product_views_30d              r = -0.287  (↓ less churn)
  avg_discount_pct_180d          r = -0.191  (↓ less churn)
  ticket_count_90d               r = -0.153  (↓ less churn)
  return_rate_180d               r = +0.069  (↑ more churn)

✅ Chart saved: charts/correlation_heatmap.png


In [20]:
# Calculate return rate per customer (using pre-snapshot orders only)
returns_per_customer = pre_snapshot.groupby('customer_id').agg(
    total_orders=('order_id', 'count'),
    total_returns=('returned', 'sum')
).reset_index()

returns_per_customer['return_rate'] = (returns_per_customer['total_returns'] / 
                                        returns_per_customer['total_orders'] * 100).round(1)

# Merge with churn labels
returns_churn = pd.merge(returns_per_customer, churn_labels[['customer_id', 'churn_next_60d']], 
                          on='customer_id')

# Split into groups: no returns, low return rate, high return rate
returns_churn['return_group'] = 'No returns'
returns_churn.loc[returns_churn['return_rate'] > 0, 'return_group'] = 'Some returns (1-30%)'
returns_churn.loc[returns_churn['return_rate'] > 30, 'return_group'] = 'High returns (>30%)'

print(" Return behaviour vs churn:\n")
return_churn_rate = returns_churn.groupby('return_group')['churn_next_60d'].agg(['mean', 'count'])
return_churn_rate.columns = ['churn_rate', 'customers']
return_churn_rate['churn_rate'] = (return_churn_rate['churn_rate'] * 100).round(1)
print(return_churn_rate)

# Overall return stats
print(f"\n Return statistics (pre-snapshot):")
print(f"  Total orders: {len(pre_snapshot)}")
print(f"  Total returns: {pre_snapshot['returned'].sum()} ({pre_snapshot['returned'].mean()*100:.1f}%)")
print(f"  Customers with at least 1 return: {(returns_per_customer['total_returns'] > 0).sum()}")


 Return behaviour vs churn:

                      churn_rate  customers
return_group                               
High returns (>30%)         61.0        187
No returns                  47.0       1932
Some returns (1-30%)        37.4        281

 Return statistics (pre-snapshot):
  Total orders: 8128
  Total returns: 534 (6.6%)
  Customers with at least 1 return: 468


In [21]:
# Merge interventions with churn labels
intervention_churn = pd.merge(interventions, churn_labels[['customer_id', 'churn_next_60d']], 
                               on='customer_id')

print("Campaign types in the dataset:")
print(intervention_churn['last_campaign_received'].value_counts().to_string())

print("\nChurn rate by CAMPAIGN TYPE:")
churn_by_campaign = intervention_churn.groupby('last_campaign_received')['churn_next_60d'].agg(['mean', 'count'])
churn_by_campaign.columns = ['churn_rate', 'customers']
churn_by_campaign['churn_rate'] = (churn_by_campaign['churn_rate'] * 100).round(1)
churn_by_campaign = churn_by_campaign.sort_values('churn_rate')
print(churn_by_campaign)

print("\n Churn rate by PRIORITY BUCKET:")
churn_by_priority = intervention_churn.groupby('manual_priority_bucket')['churn_next_60d'].agg(['mean', 'count'])
churn_by_priority.columns = ['churn_rate', 'customers']
churn_by_priority['churn_rate'] = (churn_by_priority['churn_rate'] * 100).round(1)
print(churn_by_priority)

# Campaign cost vs churn
print("\n Average campaign cost: Churners vs Stayers:")
stayed_cost = intervention_churn[intervention_churn['churn_next_60d'] == 0]['last_campaign_cost'].mean()
churned_cost = intervention_churn[intervention_churn['churn_next_60d'] == 1]['last_campaign_cost'].mean()
print(f"  Customers who stayed:  ₹{stayed_cost:.2f} avg campaign cost")
print(f"  Customers who churned: ₹{churned_cost:.2f} avg campaign cost")


Campaign types in the dataset:
last_campaign_received
none               507
new_launch         498
bundle_discount    473
free_shipping      469
welcome_offer      453

Churn rate by CAMPAIGN TYPE:
                        churn_rate  customers
last_campaign_received                       
none                          45.2        507
welcome_offer                 45.3        453
free_shipping                 46.3        469
bundle_discount               46.9        473
new_launch                    51.0        498

 Churn rate by PRIORITY BUCKET:
                        churn_rate  customers
manual_priority_bucket                       
high                          74.7       1163
low                           10.0        488
medium                        27.9        749

 Average campaign cost: Churners vs Stayers:
  Customers who stayed:  ₹18.56 avg campaign cost
  Customers who churned: ₹18.37 avg campaign cost


In [22]:
# rfm_snapshot ALREADY has churn_next_60d , But we also want to include campaign cost from interventions in the heatmap
rfm_with_interventions = pd.merge(rfm_snapshot, interventions[['customer_id', 'last_campaign_cost']], 
                                   on='customer_id')

# Select numeric columns for correlation — including campaign cost
numeric_cols = ['recency_days', 'frequency_180d', 'monetary_180d', 'return_rate_180d',
                'avg_discount_pct_180d', 'ticket_count_90d', 'sessions_30d', 
                'product_views_30d', 'last_visit_days_ago', 'last_campaign_cost', 
                'churn_next_60d']

corr_matrix = rfm_with_interventions[numeric_cols].corr()

# Chart 6: Correlation heatmap
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation Heatmap \u2014 Key Features vs Churn')

plt.tight_layout()
plt.savefig('charts/correlation_heatmap_2.png', dpi=150, bbox_inches='tight')
plt.close()

# Print the most correlated features with churn
print("\n📊 Correlation with churn_next_60d (sorted by strength):")
churn_corr = corr_matrix['churn_next_60d'].drop('churn_next_60d').sort_values(key=abs, ascending=False)
for feature, corr in churn_corr.items():
    direction = "↑ more churn" if corr > 0 else "↓ less churn"
    print(f"  {feature:<30} r = {corr:+.3f}  ({direction})")

print("\n✅ Chart saved: charts/correlation_heatmap_2.png")


📊 Correlation with churn_next_60d (sorted by strength):
  recency_days                   r = +0.561  (↑ more churn)
  last_visit_days_ago            r = +0.527  (↑ more churn)
  frequency_180d                 r = -0.394  (↓ less churn)
  monetary_180d                  r = -0.385  (↓ less churn)
  sessions_30d                   r = -0.307  (↓ less churn)
  product_views_30d              r = -0.287  (↓ less churn)
  avg_discount_pct_180d          r = -0.191  (↓ less churn)
  ticket_count_90d               r = -0.153  (↓ less churn)
  return_rate_180d               r = +0.069  (↑ more churn)
  last_campaign_cost             r = -0.007  (↓ less churn)

✅ Chart saved: charts/correlation_heatmap_2.png


In [24]:
customer_profile = pd.merge(rfm_snapshot[['customer_id', 'recency_days', 'frequency_180d', 
                                           'monetary_180d', 'return_rate_180d', 
                                           'ticket_count_90d', 'loyalty_tier', 'churn_next_60d']],
                             web_events[['customer_id', 'sessions_30d', 'product_views_30d', 
                                          'last_visit_days_ago']],
                             on='customer_id')

customer_profile = pd.merge(customer_profile, 
                             customers[['customer_id', 'age_group', 'city_tier', 
                                         'acquisition_channel']],
                             on='customer_id')

# Find interesting customer examples
# 1. High-risk churner: high recency, low activity, actually churned
churners = customer_profile[customer_profile['churn_next_60d'] == 1]
stayers = customer_profile[customer_profile['churn_next_60d'] == 0]

# Pick a churner with high recency (typical churn pattern)
typical_churner = churners.nlargest(5, 'recency_days').iloc[0]

# Pick a stayer with low recency (typical loyal customer)
typical_stayer = stayers.nsmallest(5, 'recency_days').iloc[0]

# Pick a surprising case: someone who STAYED despite high recency
surprising_stayer = stayers.nlargest(5, 'recency_days').iloc[0]

# Pick a surprising case: someone who CHURNED despite recent activity
surprising_churner = churners.nsmallest(5, 'recency_days').iloc[0]

# Display them
print("\n" + "="*70)
print("CUSTOMER-LEVEL DEEP DIVE")
print("="*70)

examples = [
    ("TYPICAL CHURNER (high risk, actually left)", typical_churner),
    ("TYPICAL STAYER (low risk, stayed loyal)", typical_stayer),
    ("SURPRISING STAYER (looked risky, but stayed)", surprising_stayer),
    ("SURPRISING CHURNER (looked safe, but left)", surprising_churner),
]

for label, cust in examples:
    print(f"\n  📌 {label}")
    print(f"     Customer ID:        {cust['customer_id']}")
    print(f"     Age group:          {cust['age_group']}, {cust['city_tier']}")
    print(f"     Acquired via:       {cust['acquisition_channel']}")
    print(f"     Loyalty tier:       {cust['loyalty_tier'] if pd.notna(cust['loyalty_tier']) else 'Not enrolled'}")
    print(f"     Recency:            {cust['recency_days']:.0f} days since last order")
    print(f"     Frequency (6mo):    {cust['frequency_180d']:.0f} orders")
    print(f"     Spending (6mo):     ₹{cust['monetary_180d']:.2f}")
    print(f"     Return rate:        {cust['return_rate_180d']*100:.1f}%")
    print(f"     Support tickets:    {cust['ticket_count_90d']:.0f}")
    print(f"     Web sessions (30d): {cust['sessions_30d']:.0f}")
    print(f"     Product views (30d):{cust['product_views_30d']:.0f}")
    print(f"     Last visit:         {cust['last_visit_days_ago']:.0f} days ago")
    print(f"     → Outcome:          {'❌ CHURNED' if cust['churn_next_60d'] == 1 else '✅ STAYED'}")

# Also show order-level detail for the typical churner
print(f"\n Order history for {typical_churner['customer_id']} (the typical churner):")
churner_orders = pre_snapshot[pre_snapshot['customer_id'] == typical_churner['customer_id']]
churner_orders = churner_orders.sort_values('order_date')
for _, order in churner_orders.iterrows():
    print(f"     {order['order_date'].strftime('%Y-%m-%d')}  |  {order['category']:12s}  |  ₹{order['gross_amount']:,.2f}  |  {'Returned' if order['returned'] else 'Kept'}")

print(f"\n  💡 Interpretation:")
print(f"     {typical_churner['customer_id']} had {len(churner_orders)} orders before the snapshot.")
print(f"     Last order was {typical_churner['recency_days']:.0f} days ago — they were already going cold.")
print(f"     {typical_churner['last_visit_days_ago']:.0f} days since last website visit — completely disengaged.")
print(f"     This is exactly the pattern our correlation analysis predicted: high recency + no web activity = churn.")



CUSTOMER-LEVEL DEEP DIVE

  📌 TYPICAL CHURNER (high risk, actually left)
     Customer ID:        CUST02309
     Age group:          25-34, Tier 2
     Acquired via:       Marketplace
     Loyalty tier:       Not enrolled
     Recency:            562 days since last order
     Frequency (6mo):    0 orders
     Spending (6mo):     ₹0.00
     Return rate:        0.0%
     Support tickets:    0
     Web sessions (30d): 1
     Product views (30d):5
     Last visit:         60 days ago
     → Outcome:          ❌ CHURNED

  📌 TYPICAL STAYER (low risk, stayed loyal)
     Customer ID:        CUST00050
     Age group:          25-34, Tier 3
     Acquired via:       Referral
     Loyalty tier:       Gold
     Recency:            0 days since last order
     Frequency (6mo):    2 orders
     Spending (6mo):     ₹2026.54
     Return rate:        50.0%
     Support tickets:    0
     Web sessions (30d): 3
     Product views (30d):12
     Last visit:         0 days ago
     → Outcome:          ✅ ST